# 01 数据获取与观测准备

这个 notebook 是目标获取和观测准备的稳定入口。它汇总 `README.md` 里的 TNS、Lasair、WISeREP 流程，并把远程下载设为手动开启，方便汇报时安全打开。

科学作用：在进入光谱分析前，先整理候选体元数据、可观测窗口、找星图、公开光谱和光变曲线。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_pipeline"
CONFIG_PATH = PROJECT_ROOT / "configs" / "sn_parameter.json"

## 配置快照

主观测流水线读取 `configs/sn_parameter.json`。TNS 和 Lasair 凭证从 `.env` 读取；`.env` 不提交到 git。

In [ ]:
config = json.loads(CONFIG_PATH.read_text())
display(config)

## 可选：刷新远程数据

平时写报告时保持 `RUN_REMOTE_FETCH = False`。只有需要重新拉取 TNS、找星图、Lasair 光变曲线和 WISeREP 光谱时，才把它改成 `True`。

In [ ]:
RUN_REMOTE_FETCH = False

if RUN_REMOTE_FETCH:
    subprocess.run([sys.executable, "scripts/fetch_target_params.py"], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, "scripts/fetch_aux_data.py"], cwd=PROJECT_ROOT, check=True)
else:
    print("跳过远程刷新，使用现有 output/ 和 data/ 产物。")

## 已有目标产物盘点

下面统计已经生成的观测报告、找星图、光变曲线、光谱和模型输出。

In [ ]:
products = []
for path in sorted(OUTPUT_DIR.glob("*")):
    if not path.is_dir() or path.name == "analysis_pipeline":
        continue
    products.append({
        "target": path.name,
        "reports": len(list(path.glob("sn_report_*.txt"))),
        "finder_charts": len(list(path.glob("finder_*"))),
        "lightcurve_files": len(list((path / "lightcurve").glob("*"))) if (path / "lightcurve").exists() else 0,
        "spectra_files": len(list((path / "spectrum").glob("*"))) if (path / "spectrum").exists() else 0,
        "superfit_files": len(list((path / "superfit").glob("*"))) if (path / "superfit").exists() else 0,
        "tardis_files": len(list((path / "tardis").glob("*"))) if (path / "tardis").exists() else 0,
    })
products_df = pd.DataFrame(products)
display(products_df)

## 光谱分析 pipeline 给出的目标状态

运行 `02_spectral_analysis_pipeline.ipynb` 或 `scripts/build_analysis_products.py` 后，这里会显示最新的目标状态表。

In [ ]:
status_path = ANALYSIS_DIR / "target_status.csv"
if status_path.exists():
    display(pd.read_csv(status_path))
else:
    print(f"缺少 {status_path}。请先运行光谱分析 pipeline。")